# A1 Tutorial - Machine Learning: Housing Price Prediction
## Introduction
**In-class activity:** Housing price prediction

**Assignment A1:** Improve the housing price model by adding features, comparing two models, and explaining error sources.

**Data source:** [King County house sales CSV](https://raw.githubusercontent.com/yantaolab/Autonomous-Construction-and-Robotics/refs/heads/main/Projects/Project%2001%20House%20Price%20Prediction/A1_Housing_Price_data.csv)

The dataset records home sales in King County, including Seattle, between May 2014 and May 2015. The exercise is intentionally designed to fit within a 60-minute tutorial.


## Learning outcomes

By the end of this tutorial, you should be able to:

1. Inspect a real housing-price dataset and identify simple data-quality or modeling issues.
2. Build a baseline linear regression model.
3. Improve the model by adding meaningful features.
4. Compare two models using test R², MAE, and RMSE.
5. Explain at least two sources of prediction error using evidence from plots or tables.

> **Important:** A1 is an **in-class assignment**. Complete the final template and export your work to PDF before leaving.


## 60-minute flow

| Time | Activity | Student output |
|---|---|---|
| 0–5 min | Objectives and context | One prediction about useful features |
| 5–15 min | Inspect and visualise the data | Two observations |
| 15–25 min | Build Model A | Baseline metrics and plot |
| 25–40 min | Complete A1 feature task | Feature list and reasons |
| 40–50 min | Train Model B | Comparison table |
| 50–58 min | Diagnose errors | Two evidence-based error sources |
| 58–60 min | Submit | Completed PDF |


In [ ]:
# Environment setup
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 3
TEST_SIZE = 0.20
TARGET = "price"
print("Environment ready.")


In [ ]:
# Load and prepare the real dataset
DATA_URL = "https://raw.githubusercontent.com/yantaolab/Autonomous-Construction-and-Robotics/refs/heads/main/Projects/Project%2001%20House%20Price%20Prediction/A1_Housing_Price_data.csv"
LOCAL_PATHS = [
    Path(r"A1_Housing_Price_data.csv"),
    Path(r"sample_data/A1_Housing_Price_data.csv"),
]

for path in LOCAL_PATHS:
    if path.exists():
        df = pd.read_csv(path)
        data_source = str(path)
        break
else:
    df = pd.read_csv(DATA_URL)
    data_source = DATA_URL

# Derived features are calculated only from information already in the dataset.
df["sale_date"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%S")
df["house_age"] = df["sale_date"].dt.year - df["yr_built"]
df["renovated"] = (df["yr_renovated"] > 0).astype(int)

print("Loaded from:", data_source)
print("Rows and columns after adding two derived columns:", df.shape)
print("Original CSV rows:", len(df))
print("Missing values:", int(df.isna().sum().sum()))
display(df.head(3))


### Data-reading note

The CSV has 21,613 observations and 21 original columns. The notebook adds `sale_date`, `house_age`, and `renovated` for teaching purposes. The target is `price`; `id` is an identifier and is not used as a predictor.


In [ ]:
# Quick inspection
key_columns = [
    "price", "sqft_living", "bedrooms", "bathrooms",
    "grade", "condition", "waterfront", "view",
    "house_age", "lat", "long",
]
display(df[key_columns].describe().T.round(2))
print("Condition counts:")
display(df["condition"].value_counts().sort_index())


In [ ]:
# Visualisation
plot_df = df.sample(n=min(3000, len(df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.scatterplot(data=plot_df, x="sqft_living", y="price", alpha=0.25, edgecolor=None, ax=axes[0])
axes[0].set_title("House price vs. living area")
axes[0].set_xlabel("Living area (sq ft)")
axes[0].set_ylabel("Sale price (USD)")
axes[0].ticklabel_format(style="plain", axis="y")
sns.histplot(data=df, x="price", bins=50, kde=True, color="#4C72B0", ax=axes[1])
axes[1].set_title("Distribution of sale prices")
axes[1].set_xlabel("Sale price (USD)")
axes[1].set_ylabel("Count")
axes[1].ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()


In [ ]:
# Check possible data issues
print("Largest bedroom values:")
display(df.nlargest(3, "bedrooms")[["id", "price", "bedrooms", "bathrooms", "sqft_living", "grade"]])
print("Rows belonging to repeated property IDs:", int(df["id"].duplicated(keep=False).sum()))
area_identity = (df["sqft_living"] == df["sqft_above"] + df["sqft_basement"]).all()
print("sqft_living = sqft_above + sqft_basement for every row:", area_identity)


In [ ]:
# Use one fixed split for both models
train_df, test_df = train_test_split(df, test_size=0.2, random_state=3)
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


In [ ]:
# Reusable model evaluation function
def fit_and_evaluate(model_name, features):
    model = LinearRegression()
    model.fit(train_df[features], train_df[TARGET])
    pred_train = model.predict(train_df[features])
    pred_test = model.predict(test_df[features])
    metrics = {
        "Model": model_name,
        "No_of_features": len(features),
        "Train_R2": r2_score(train_df[TARGET], pred_train),
        "Test_R2": r2_score(test_df[TARGET], pred_test),
        "Test_MAE_USD": mean_absolute_error(test_df[TARGET], pred_test),
        "Test_RMSE_USD": np.sqrt(mean_squared_error(test_df[TARGET], pred_test)),
    }
    return {"name": model_name, "features": features, "model": model, "pred_train": pred_train, "pred_test": pred_test, "metrics": metrics}


### Pair discussion (1 minute)

- Which variable would you use first to build a baseline? (P12 & P16)


In [ ]:
# Model A: baseline
baseline_features = ["sqft_living"]
model_A = fit_and_evaluate("Model A: Baseline", baseline_features)
display(pd.DataFrame([model_A["metrics"]]).round({"Train_R2": 3, "Test_R2": 3, "Test_MAE_USD": 0, "Test_RMSE_USD": 0}))


### Metric reminder

- **R²:** proportion of test-target variation explained by the model; higher is better, but it is not enough on its own.
- **MAE:** average absolute price error; lower is better and it is in USD here.
- **RMSE:** square-root mean squared error; lower is better and large errors receive more weight.


In [ ]:
# Baseline regression plot
x_test_A = test_df["sqft_living"].to_numpy()
y_test = test_df[TARGET].to_numpy()
pred_A = model_A["pred_test"]
sort_index = np.argsort(x_test_A)
plt.figure(figsize=(9.2, 6.1))
plt.scatter(x_test_A, y_test, alpha=0.25, color="#2E7D32", label="Actual test data")
plt.plot(x_test_A[sort_index], pred_A[sort_index], color="#C62828", linewidth=2.2, label="Baseline prediction")
plt.xlabel("Living area (sq ft)")
plt.ylabel("Sale price (USD)")
plt.title("Model A: simple linear regression")
plt.ticklabel_format(style="plain", axis="y")
plt.legend()
plt.tight_layout()
plt.show()


## A1 — In-class assignment

### Improve the housing price model by adding features, comparing two models, and explaining error sources

**Task:** Add at least **four** features to Model A. You may choose from the list below, or propose another numeric feature that you can justify.

`bedrooms`, `bathrooms`, `sqft_lot`, `floors`, `waterfront`, `view`, `condition`, `grade`, `house_age`, `renovated`, `lat`, `long`, `sqft_living15`, `sqft_lot15`

**Requirements:**
1. Use the same train/test split as Model A.
2. Do not use `price` or `id` as a predictor.
3. Record the added features and give a one-sentence reason for each.
4. Compare Model A and Model B using test R², MAE, and RMSE.
5. Explain at least two error sources with evidence from a plot or the largest-error table.

> **Interaction:** Work in pairs for 5 minutes. Each pair must defend one feature choice.


### Pair discussion (1 minute)

- Why should a model comparison use the same train/test split? (P18)


In [ ]:
# Student starter cell — edit added_features during class
candidate_features = [
    "bedrooms", "bathrooms", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade",
    "house_age", "renovated", "lat", "long",
    "sqft_living15", "sqft_lot15",
]

# Replace or edit this list. At least four features are required.
added_features = [
# please adding the features you want 
]

if len(added_features) < 4:
    raise ValueError("Please add at least four features.")
if len(set(added_features)) != len(added_features):
    raise ValueError("A feature was repeated.")
invalid_features = [f for f in added_features if f not in candidate_features]
if invalid_features:
    raise ValueError(f"Invalid features: {invalid_features}")

improved_features = baseline_features + added_features
print("Added features:", added_features)
print("Model B features:", improved_features)


In [ ]:
# Model B and comparison
model_B = fit_and_evaluate("Model B: Improved", improved_features)
comparison = pd.DataFrame([model_A["metrics"], model_B["metrics"]])
display(comparison.round({"Train_R2": 3, "Test_R2": 3, "Test_MAE_USD": 0, "Test_RMSE_USD": 0}))

delta_r2 = model_B["metrics"]["Test_R2"] - model_A["metrics"]["Test_R2"]
mae_reduction = (model_A["metrics"]["Test_MAE_USD"] - model_B["metrics"]["Test_MAE_USD"]) / model_A["metrics"]["Test_MAE_USD"] * 100
rmse_reduction = (model_A["metrics"]["Test_RMSE_USD"] - model_B["metrics"]["Test_RMSE_USD"]) / model_A["metrics"]["Test_RMSE_USD"] * 100
print(f"Change in test R²: {delta_r2:+.3f}")
print(f"MAE reduction: {mae_reduction:+.1f}%")
print(f"RMSE reduction: {rmse_reduction:+.1f}%")


In [ ]:
# Diagnostic plots for Model B
pred_B = model_B["pred_test"]
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.6))
lower, upper = min(y_test.min(), pred_B.min()), max(y_test.max(), pred_B.max())
axes[0].scatter(y_test, pred_B, alpha=0.25, color="#4C72B0")
axes[0].plot([lower, upper], [lower, upper], "--", color="#C62828")
axes[0].set_title("Model B: actual vs. predicted")
axes[0].set_xlabel("Actual price")
axes[0].set_ylabel("Predicted price")
axes[0].ticklabel_format(style="plain", axis="both")
residuals = y_test - pred_B
axes[1].scatter(pred_B, residuals, alpha=0.25, color="#55A868")
axes[1].axhline(0, color="#C62828", linestyle="--")
axes[1].set_title("Model B residuals")
axes[1].set_xlabel("Predicted price")
axes[1].set_ylabel("Residual = actual − predicted")
axes[1].ticklabel_format(style="plain", axis="both")
plt.tight_layout()
plt.show()


In [ ]:
# Largest Model B errors
error_table = test_df[["id", "price", "sqft_living", "grade", "waterfront", "view", "house_age", "lat", "long"]].copy()
error_table["prediction_B"] = model_B["pred_test"]
error_table["residual_B"] = error_table["price"] - error_table["prediction_B"]
error_table["absolute_error_B"] = error_table["residual_B"].abs()
display(error_table.nlargest(10, "absolute_error_B").round(2))
print("Use the table as evidence: what features or situations are common among the largest errors?")


### What counts as an error source? (P19)

Do not call every unusual observation a data error. Distinguish among:

- **outliers:** observations far from the main pattern;
- **omitted variables:** important information not included, such as school quality or interior finish;
- **non-linearity:** the true relationship is not well represented by one straight line;
- **spatial heterogeneity:** the relationship differs across locations;
- **sampling or measurement limitations:** the data cover a particular place and period.


## Submission template

### 1. Features added
List at least four features and give a reason for each.

### 2. Model comparison
Complete the table with test-set values.

| Model | Test R² | Test MAE (USD) | Test RMSE (USD) |
|---|---:|---:|---:|
| Model A: Baseline | | | |
| Model B: Improved | | | |

### 3. Better model
The better model is __________ because __________.

### 4. Error source 1
Claim: __________. Evidence: __________.

### 5. Error source 2
Claim: __________. Evidence: __________.

**Submission:** Export the complete notebook and answer sheet as a PDF and submit it in class.


## References

Rafiei, M. H., & Adeli, H. (2016). A novel machine learning model for estimation of sale prices of real estate units. *Journal of Construction Engineering and Management, 142*(10), 04016053. https://doi.org/10.1061/(ASCE)CO.1943-7862.0001047

Soltani, A., Heydari, M., Aghaei, F., & Pettit, C. J. (2022). Housing price prediction incorporating spatio-temporal dependency into machine learning algorithms. *Cities, 131*, 103941. https://doi.org/10.1016/j.cities.2022.103941

Calainho, F. D., van de Minne, A. M., & Francke, M. K. (2024). A machine learning approach to price indices: Applications in commercial real estate. *Journal of Real Estate Finance and Economics, 68*(4), 1308–1339. https://doi.org/10.1007/s11146-022-09893-1

Lee, H., & Han, H. (2024). Machine learning approach to residential valuation: A convolutional neural network model for geographic variation. *The Annals of Regional Science, 72*(2), 565–592. https://doi.org/10.1007/s00168-023-01212-7

Chiu, S.-M., Chen, Y.-C., & Lee, C. (2022). Estate price prediction system based on temporal and spatial features and lightweight deep learning model. *Applied Intelligence, 52*, 808–834. https://doi.org/10.1007/s10489-021-02472-6
